# 03 — Regional zooms at native resolution

Surface heat flux structure at the ~2–4 km native LLC2160 resolution in two western
boundary current regions, where mesoscale and submesoscale SST gradients imprint strongly
on air–sea fluxes:

- **Gulf Stream** (82°W–50°W, 25°N–45°N)
- **Kuroshio / Kuroshio Extension** (125°E–165°E, 25°N–45°N)

Faces intersecting each box are plotted directly with `pcolormesh` — no regridding.

In [ ]:
# Environment check: run on SciServer (Kraken domain, with the Poseidon DYAMOND
# ceph volume attached), or set DYAMOND_ROOT to a local subset.
# SciServer containers do not persist `pip install --user` across restarts, so
# fall back to importing directly from the repo's src/ tree if needed.
try:
    from dyamond_fluxes import dyamond_root
except ModuleNotFoundError:
    import sys
    from pathlib import Path as _P

    sys.path.insert(0, str((_P.cwd() / ".." / "src").resolve()))
    from dyamond_fluxes import dyamond_root

root = dyamond_root()
print(f"DYAMOND root: {root}")

In [ ]:
from dyamond_fluxes import nonsolar_flux, open_ocean_dataset, to_positive_down

ds = open_ocean_dataset(["oceQnet", "oceQsw"])

# A boreal-winter snapshot: strongest turbulent heat loss over the western
# boundary currents (cold-air outbreaks).
SNAPSHOT = "2021-01-15T00:00"
snap = ds.sel(time=SNAPSHOT, method="nearest")
print("snapshot:", snap.time.values)

qnet = to_positive_down(snap["oceQnet"])
qsw = to_positive_down(snap["oceQsw"])
qns = nonsolar_flux(qnet, qsw)
ocean = ds["Depth"] > 0
qnet, qns = qnet.where(ocean), qns.where(ocean)

In [ ]:
from pathlib import Path

from dyamond_fluxes.plotting import plot_region

FIGDIR = Path("../figures")
FIGDIR.mkdir(exist_ok=True)

REGIONS = {
    "gulf_stream": (-82.0, -50.0, 25.0, 45.0),
    "kuroshio": (125.0, 165.0, 25.0, 45.0),
}
lon, lat = ds["XC"], ds["YC"]

In [ ]:
for name, bbox in REGIONS.items():
    fig, ax = plot_region(
        qnet.load(), lon, lat, bbox,
        title=f"Net surface heat flux, {name.replace('_', ' ')}, {str(snap.time.values)[:16]}",
    )
    fig.savefig(FIGDIR / f"qnet_{name}.png", dpi=200, bbox_inches="tight")

In [ ]:
for name, bbox in REGIONS.items():
    fig, ax = plot_region(
        qns.load(), lon, lat, bbox,
        title=f"Non-solar heat flux, {name.replace('_', ' ')}, {str(snap.time.values)[:16]}",
    )
    fig.savefig(FIGDIR / f"qns_{name}.png", dpi=200, bbox_inches="tight")

In winter, the non-solar flux over the Gulf Stream and Kuroshio should show intense ocean
heat loss (large negative $Q_{ns}$, locally beyond $-800$ W m$^{-2}$ during cold-air
outbreaks) organized along the current cores and warm-core rings — structure that only
emerges at kilometer-scale resolution. Compare against a summer snapshot by changing
`SNAPSHOT`.